# 07 Attention on DNA: a transformer encoder finds the motif

Lecture 11, architectures and representations. Book chapter 10 (attention and
the transformer encoder).

**Goals**

- Turn DNA into tokens, add positional encodings, and classify with a
  one-layer transformer encoder through a CLS token.
- See that the choice of tokens matters: single bases fail, overlapping
  3-mers work.
- Read the attention of the CLS token: where does the model look?
- Compare with a hybrid model (a `Conv1d` stem before the encoder) and with
  the CNN of notebook 06.

**Expected findings** (asserted in `tests/test_examples.py::test_07_*`):

1. With single-base tokens the transformer stays at chance (test accuracy
   below 0.6); with 3-mer tokens it exceeds 0.8, and the CLS token puts more
   than three times the uniform share of its attention on the six 3-mers of
   the motif.
2. A `Conv1d` stem before the encoder reaches more than 0.95, like the CNN.

The data are those of notebook 06: 1000 training and 1000 test sequences of
length 60, half of them with the motif `TGACGTCA`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dna_motifs import kmer_tokens, make_motif_dataset, one_hot

import glassnn.functional as F
from glassnn import Tensor, manual_seed, nn, no_grad, optim
from glassnn.data import DataLoader

sequences, labels, positions = make_motif_dataset(2000, seed=1)
train_s, test_s = sequences[:1000], sequences[1000:]
train_y, test_y, test_positions = labels[:1000], labels[1000:], positions[1000:]

## Tokens and the model

A sequence becomes a list of token ids. A CLS ("classification") token is
put in front; its output vector, after the encoder, is the summary used by
the classifier. Every token is embedded (`nn.Embedding`), the sinusoidal
positional encoding is added (attention by itself ignores the order), and
one pre-LN encoder layer with two heads follows.

In [ ]:
def with_cls(tokens, cls_id):
    return np.hstack([np.full((len(tokens), 1), cls_id), tokens])


class TokenTransformer(nn.Module):
    def __init__(self, vocabulary, d_model=32):
        super().__init__()
        self.embed = nn.Embedding(vocabulary, d_model)
        self.position = nn.PositionalEncoding(d_model, max_len=100)
        layer = nn.TransformerEncoderLayer(
            d_model, nhead=2, dim_feedforward=64, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=1, norm=nn.LayerNorm(d_model)
        )
        self.head = nn.Linear(d_model, 2)

    def forward(self, tokens):
        return self.head(self.encoder(self.position(self.embed(tokens)))[:, 0])

    def cls_attention(self, tokens):
        # Attention weights of the CLS query (row 0), averaged over the heads.
        layer = self.encoder.layers[0]
        with no_grad():
            h = layer.norm1(self.position(self.embed(tokens)))
            _, weights = layer.self_attn(h, h, h)
        return weights.data[:, 0]


def fit(model, X, y, epochs=10, lr=3e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for xb, yb in DataLoader(X, y, batch_size=32, shuffle=True):
            optimizer.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            optimizer.step()
    return model


def accuracy(model, X, y):
    with no_grad():
        return float((model(Tensor(X)).data.argmax(axis=1) == y).mean())

## Single bases against 3-mers

In [ ]:
single = [with_cls(kmer_tokens(s, 1), 4) for s in (train_s, test_s)]
manual_seed(0)
model_1 = fit(TokenTransformer(5), single[0], train_y)
print("single-base tokens: test accuracy", accuracy(model_1, single[1], test_y))

triple = [with_cls(kmer_tokens(s, 3), 64) for s in (train_s, test_s)]
manual_seed(0)
model_3 = fit(TokenTransformer(65), triple[0], train_y)
print("3-mer tokens:       test accuracy", accuracy(model_3, triple[1], test_y))

With single bases, the motif is a conjunction of 8 tokens at consecutive
positions. One attention layer computes, for the CLS query, a weighted
*average* of the value vectors; an average of single-base embeddings says
little about which bases are adjacent, and the network does not find the
motif. A 3-mer token already carries three adjacent bases, and the six
3-mers inside the motif are rare in random sequence, so attending to them is
informative. Tokenization is part of the model: large DNA language models
use k-mers or learned sub-word tokens for this reason.

## Where does the CLS token look?

In [ ]:
positives = np.flatnonzero(test_y == 1)
weights = model_3.cls_attention(Tensor(triple[1][positives]))
starts = test_positions[positives] + 1  # +1: the CLS token comes first
on_motif = np.array([w[p : p + 6].sum() for w, p in zip(weights, starts, strict=True)])
print(
    f"mean attention on the motif's 3-mers: {on_motif.mean():.2f} "
    f"(uniform share: {6 / weights.shape[1]:.2f})"
)

fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
for ax, j in zip(axes, range(3), strict=True):
    ax.bar(range(weights.shape[1]), weights[j])
    ax.axvspan(starts[j] - 0.5, starts[j] + 5.5, color="orange", alpha=0.3)
    ax.set_ylabel("weight")
axes[-1].set_xlabel("token (0 = CLS); orange: the motif's 3-mers")
plt.show()

The CLS token puts about 40 % of its attention on the six 3-mers of the
motif, four times their uniform share. The other peaks are not noise: the
model attends to certain 3-mers of the motif (above all `TCA`) *wherever*
they occur, and copies of them outside the motif get attention too. One
attention layer cannot check that the six 3-mers are adjacent, which is why
the accuracy stops near 0.88, while the CNN filter of notebook 06 sees all
eight bases at once.

## A convolutional stem

A hybrid model lets a convolution build local features (width 5) and the
encoder relate them across the sequence; the output is averaged over
positions instead of using a CLS token.

In [ ]:
class HybridTransformer(nn.Module):
    def __init__(self, d_model=32):
        super().__init__()
        self.stem = nn.Conv1d(4, d_model, 5, padding="same")
        self.position = nn.PositionalEncoding(d_model, max_len=100)
        layer = nn.TransformerEncoderLayer(
            d_model, nhead=2, dim_feedforward=64, dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=1, norm=nn.LayerNorm(d_model)
        )
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        h = self.position(self.stem(x).transpose(1, 2))  # (N, L, d)
        return self.head(self.encoder(h).mean(dim=1))


manual_seed(0)
hybrid = fit(HybridTransformer(), one_hot(train_s), train_y)
print("Conv1d stem + encoder: test accuracy", accuracy(hybrid, one_hot(test_s), test_y))

**Findings.**

1. Single-base tokens: chance level. 3-mer tokens: well above chance; the
   CLS token attends to the 3-mers of the motif (about four times their
   uniform share), including copies of them elsewhere in the sequence. The
   attention weights point to the evidence the model uses, which is not
   the same as a complete explanation of the prediction.
2. The hybrid model is nearly as accurate as the CNN of notebook 06 (0.98
   against 0.996):
   local patterns are what convolutions do best; attention adds the
   relations between distant positions, which this task does not need.

**Exercises.**

1. Train the 3-mer model with two encoder layers. Does accuracy improve?
   What do the attention maps of the second layer show?
2. Remove the positional encoding from the 3-mer model. What changes, and
   why does the model still work? (Hint: a 3-mer is local by itself.)
3. Plant *two* motifs and label a sequence positive only if they are at
   most 10 bases apart. Compare the CNN of notebook 06 with the hybrid
   model.